<a href="https://colab.research.google.com/github/fazilooffsaid/defor-web-/blob/main/week6_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6: Hyperparameter Tuning & Regularization Techniques for Deep Learning Models

**Course:** Hyperparameter Tuning and Regularization Techniques for Generative AI Models

---

## Learning Objectives (Session Objectives)

By the end of this lab, you will be able to:

1. **Apply systematic hyperparameter tuning** (e.g., grid search) to improve deep learning model performance and understand the impact of learning rate, batch size, and architecture choices.

2. **Implement and compare regularization techniques** including Dropout, L2 weight decay, and Batch Normalization to reduce overfitting and improve generalization.

3. **Interpret training and validation curves** to identify overfitting, underfitting, and the effect of different hyperparameters and regularization strategies.

4. **Design and run reproducible experiments** using PyTorch on standard datasets (MNIST, Fashion-MNIST) with clear metrics, visualizations, and model checkpointing.

5. **Synthesize findings** by selecting the best model configuration and writing concise explanations of why certain hyperparameters or regularization techniques worked best.

---
## Part I – Hyperparameter Tuning Example (MNIST)

In this part we load MNIST, define a simple MLP, implement a reusable training function, and run **grid search** over learning rate, batch size, and hidden layer sizes. We then compare the best and worst configurations with convergence curves.

### 1.1 Import Libraries

In [ ]:
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

# Device: use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Dark-theme friendly plotting style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except Exception:
    plt.style.use('seaborn-darkgrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 10

We import PyTorch (`torch`, `nn`, `optim`), `DataLoader`, and **torchvision** for datasets and transforms. Matplotlib, NumPy, pandas, and **tqdm** are used for plotting, numerical work, results tables, and progress bars. The **device** is set to CUDA if a GPU is available, otherwise CPU—all model and tensor operations will use this device. Plotting is configured with a dark-theme-friendly style (`seaborn-v0_8-darkgrid`) and default figure size and font size for consistent, readable plots.

### 1.2 Load MNIST Dataset

In [ ]:
# MNIST mean and std (standard normalization values for MNIST)
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

# Download and load train and test sets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Use 10% of training data as validation (for grid search)
n_train = len(train_dataset)
n_val = max(1, n_train // 10)
n_train_use = n_train - n_val
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [n_train_use, n_val])

print(f"Train size: {len(train_subset)}, Val size: {len(val_subset)}, Test size: {len(test_dataset)}")

We define a **transform** pipeline: `ToTensor()` converts PIL images to tensors in range [0, 1], and `Normalize(0.1307, 0.3081)` subtracts the MNIST mean and divides by the std so inputs have zero mean and unit variance—this speeds up and stabilizes training. We load the **train** and **test** MNIST splits with this transform. We then split the official training set into **train (90%)** and **validation (10%)** using `random_split`; validation is used during grid search to pick the best hyperparameters without touching the test set.

### 1.3 Dataset Description & Visualization

In [ ]:
# --- Dataset description ---
num_classes = 10
class_names = [str(i) for i in range(num_classes)]
sample_img, _ = train_dataset[0]
img_shape = sample_img.shape  # (C, H, W)

print("=== MNIST Dataset Description ===")
print(f"Total training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")
print(f"Class labels: {class_names}")
print(f"Image shape (C, H, W): {img_shape}")
print(f"Flattened input size: {img_shape[0] * img_shape[1] * img_shape[2]}")

 We print a **dataset description**: total training and test sample counts, number of classes (10 digits 0–9), class labels, and the **image shape** (C, H, W). For MNIST this is (1, 28, 28). The flattened input size 784 = 1×28×28 is the number of features the MLP will receive per image.

In [ ]:
# --- 2x5 grid: 10 sample images with class labels ---
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.flatten()
for i in range(10):
    img, label = train_dataset[i]
    axes[i].imshow(img.squeeze(), cmap='gray')
    axes[i].set_title(f'Label: {label}', fontsize=11)
    axes[i].axis('off')
plt.suptitle('MNIST – 10 Sample Images (2×5)', fontsize=12)
plt.tight_layout()
plt.show()

 We create a **2×5 subplot grid** and display the **first 10 training images** with their **class labels** (0–9). Each image is shown with `imshow` on the squeezed tensor (28×28); the title shows the true label. This gives a quick visual check of the data and normalization.

In [ ]:
# --- One example per class in a nice grid ---
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.flatten()
for c in range(10):
    idx = next(i for i in range(len(train_dataset)) if train_dataset[i][1] == c)
    img, label = train_dataset[idx]
    axes[c].imshow(img.squeeze(), cmap='gray')
    axes[c].set_title(f'Class {c}', fontsize=11)
    axes[c].axis('off')
plt.suptitle('MNIST – One Example per Class', fontsize=12)
plt.tight_layout()
plt.show()

For **each class 0–9** we find the **first training index** that has that label and display that image in a 2×5 grid. So we get **exactly one representative image per digit**, helping us see how each class looks and whether the dataset is balanced.

In [ ]:
# --- Class distribution bar chart ---
train_labels = [train_dataset[i][1] for i in range(len(train_dataset))]
counts = np.bincount(train_labels, minlength=10)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_names, counts, color=plt.cm.viridis(np.linspace(0.2, 0.8, 10)))
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_title('MNIST – Class Distribution (Training Set)')
for b, v in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 50, str(v), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

 We collect all **training labels**, use `np.bincount` to get the **count per class**, and draw a **bar chart** with class names on the x-axis and counts on the y-axis. Bars are colored with a viridis gradient and counts are annotated on top. This shows whether the dataset is **balanced** (roughly equal samples per class) or skewed.

### 1.4 Define Simple MLP

In [ ]:
class SimpleMLP(nn.Module):
    """MLP: input 28×28=784 → hidden layers → 10 output (num_classes)."""
    def __init__(self, input_size=784, hidden_sizes=(128, 64), num_classes=10, dropout_rate=0.0):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            prev = h
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Linear(prev, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return self.classifier(x)

We define **SimpleMLP**, a feedforward network. The input is flattened to 784. A sequence of **hidden layers** is built from `hidden_sizes`: each layer is Linear → ReLU, and optionally **Dropout** (when `dropout_rate > 0`). The last layer (**classifier**) maps the final hidden size to **10 classes** (digits). The `forward` method flattens the input batch, passes it through `features`, then through `classifier` to produce logits for classification.

### 1.5 Reusable Training Function

In [ ]:
def train_model(train_loader, val_loader, test_loader, learning_rate, batch_size, hidden_sizes,
                epochs=5, dropout_rate=0.0, weight_decay=0.0, use_batchnorm=False, verbose=True):
    """
    Train an MLP and return train/val loss and accuracy per epoch, plus final test accuracy.
    """
    # Build model (we use SimpleMLP; if use_batchnorm we need a variant – see Part II)
    model = SimpleMLP(784, hidden_sizes, 10, dropout_rate=dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False, disable=not verbose)
        for X, y in pbar:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            pbar.set_postfix(loss=loss.item(), acc=correct/total)
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                logits = model(X)
                val_loss += criterion(logits, y).item()
                val_correct += (logits.argmax(dim=1) == y).sum().item()
                val_total += y.size(0)
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(val_correct / val_total)

    # Final test accuracy
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            test_correct += (model(X).argmax(dim=1) == y).sum().item()
            test_total += y.size(0)
    test_acc = test_correct / test_total
    return history, test_acc, model

We define a **reusable `train_model` function** that: (1) builds a **SimpleMLP** with the given `hidden_sizes` and `dropout_rate`, (2) uses **Adam** with the given `learning_rate` and optional `weight_decay`, (3) trains for `epochs` with **CrossEntropyLoss**. Each epoch: we run over the train loader (with tqdm), compute loss, backprop, and record **train loss and accuracy**; then we evaluate on the validation loader (no grad) and record **val loss and accuracy**. At the end we compute **final test accuracy** over the test loader. The function returns the **history** (lists of train/val loss and acc per epoch), **test accuracy**, and the **trained model**.

### 1.6 Grid Search

In [ ]:
# Grid search: lr, batch_size, hidden_sizes; 5 epochs each
lr_list = [0.001, 0.01, 0.1]
batch_size_list = [32, 64, 128]
hidden_sizes_list = [[128, 64], [256, 128]]
epochs = 5

results = []
all_histories = {}  # (lr, bs, tuple(hidden)) -> history

for lr in lr_list:
    for batch_size in batch_size_list:
        for hidden_sizes in hidden_sizes_list:
            key = (lr, batch_size, tuple(hidden_sizes))
            train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0)
            val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
            test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
            print(f"\n--- lr={lr}, batch_size={batch_size}, hidden_sizes={hidden_sizes} ---")
            history, test_acc, _ = train_model(train_loader, val_loader, test_loader,
                                               learning_rate=lr, batch_size=batch_size,
                                               hidden_sizes=hidden_sizes, epochs=epochs, dropout_rate=0.0)
            all_histories[key] = history
            results.append({
                'lr': lr, 'batch_size': batch_size, 'hidden_sizes': str(hidden_sizes),
                'train_acc_final': history['train_acc'][-1], 'val_acc_final': history['val_acc'][-1],
                'test_acc': test_acc
            })

df_results = pd.DataFrame(results)
print("\n=== Grid Search Results ===")
display(df_results)

We run **grid search** over all combinations of: **learning rate** (0.001, 0.01, 0.1), **batch size** (32, 64, 128), and **hidden_sizes** ([128,64], [256,128]). For each combination we create new data loaders, call `train_model` for 5 epochs (no dropout), and store the **history** in `all_histories` and the **final train acc, val acc, and test acc** in a results list. The results are shown in a **pandas DataFrame** so we can compare which configuration performed best on validation (and test).

In [ ]:
# Identify best and worst by validation accuracy
best_row = df_results.loc[df_results['val_acc_final'].idxmax()]
worst_row = df_results.loc[df_results['val_acc_final'].idxmin()]

best_key = (best_row['lr'], int(best_row['batch_size']), tuple(eval(best_row['hidden_sizes'])))
worst_key = (worst_row['lr'], int(worst_row['batch_size']), tuple(eval(worst_row['hidden_sizes'])))

print(f"Best config: lr={best_key[0]}, batch_size={best_key[1]}, hidden_sizes={best_key[2]}")
print(f"Worst config: lr={worst_key[0]}, batch_size={worst_key[1]}, hidden_sizes={worst_key[2]}")

From the results DataFrame we take the row with **maximum validation accuracy** (best config) and **minimum validation accuracy** (worst config). We build the same **key** (lr, batch_size, tuple(hidden_sizes)) used when storing histories, so we can later plot the training and validation curves for these two configurations.

### 1.7 Convergence Curves: Best vs Worst

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, epochs + 1)

for key, label in [(best_key, 'Best'), (worst_key, 'Worst')]:
    h = all_histories[key]
    axes[0].plot(epochs_range, h['train_loss'], label=f'{label} (train)')
    axes[0].plot(epochs_range, h['val_loss'], '--', label=f'{label} (val)')
    axes[1].plot(epochs_range, h['train_acc'], label=f'{label} (train)')
    axes[1].plot(epochs_range, h['val_acc'], '--', label=f'{label} (val)')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss: Best vs Worst Configuration')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy: Best vs Worst Configuration')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

We plot **two figures side by side**: (1) **Loss** vs epoch for the best and worst configurations—both train and validation loss; (2) **Accuracy** vs epoch for the same. Solid lines are training, dashed are validation. Comparing best vs worst shows how hyperparameters affect **convergence speed**, **final performance**, and the **train–val gap** (overfitting).

### 1.8 Conclusion: What We Learned About Hyperparameter Impact

- **Learning rate**: Too high (e.g. 0.1) can cause unstable training or poor convergence; too low can be slow. A moderate value (e.g. 0.01 or 0.001) often works best.
- **Batch size**: Smaller batches (e.g. 32) give noisier gradients but can generalize better; larger batches (128) can speed up training but may converge to sharper minima.
- **Hidden sizes**: Deeper/wider networks (e.g. [256, 128]) have more capacity but may overfit if not regularized; [128, 64] can be sufficient for MNIST.
- **Grid search** helps us compare combinations systematically and pick the best configuration for validation (and typically test) performance.

---
## Part II – Regularization Techniques (MNIST)

We continue with the **best model configuration from Part I**. We train four versions: (a) baseline, (b) Dropout (p=0.2 and p=0.5), (c) L2 weight decay (1e-4 and 1e-3), (d) BatchNorm + Dropout. Then we compare training curves and overfitting gap.

### 2.1 MLP with Optional BatchNorm

For (d) we need an MLP that supports BatchNorm; we define a small variant.

In [ ]:
class MLPWithBatchNorm(nn.Module):
    """MLP with BatchNorm after each hidden layer (stabilizes training, acts as regularizer)."""
    def __init__(self, input_size=784, hidden_sizes=(128, 64), num_classes=10, dropout_rate=0.0):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            prev = h
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Linear(prev, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return self.classifier(x)

 We define **MLPWithBatchNorm**, an MLP that inserts **BatchNorm1d** after each linear layer (before ReLU). BatchNorm normalizes the activations of each layer over the batch (zero mean, unit variance), then applies learnable scale and shift. This **stabilizes training** and often acts as a mild regularizer. The rest of the structure (optional Dropout, classifier head) is the same as SimpleMLP.

### 2.2 Training Function Supporting BatchNorm and Weight Decay

We use the same `train_model` from Part I but pass `weight_decay` to the optimizer. For BatchNorm we need a separate training loop that uses `MLPWithBatchNorm`; we add a helper that trains and returns history + test_acc.

In [ ]:
def train_model_reg(train_loader, val_loader, test_loader, learning_rate, batch_size, hidden_sizes,
                    epochs=6, dropout_rate=0.0, weight_decay=0.0, use_batchnorm=False):
    """Train MLP (with or without BatchNorm) and return history + final test accuracy."""
    if use_batchnorm:
        model = MLPWithBatchNorm(784, hidden_sizes, 10, dropout_rate=dropout_rate).to(device)
    else:
        model = SimpleMLP(784, hidden_sizes, 10, dropout_rate=dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X, y in tqdm(train_loader, desc=f'Epoch {epoch+1}', leave=False):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)
        history['train_loss'].append(running_loss / len(train_loader))
        history['train_acc'].append(correct / total)
        model.eval()
        vloss, vcorr, vtot = 0.0, 0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                vloss += criterion(out, y).item()
                vcorr += (out.argmax(dim=1) == y).sum().item()
                vtot += y.size(0)
        history['val_loss'].append(vloss / len(val_loader))
        history['val_acc'].append(vcorr / vtot)

    model.eval()
    tcorr, ttot = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            tcorr += (model(X).argmax(dim=1) == y).sum().item()
            ttot += y.size(0)
    return history, tcorr / ttot

# Fix best config from Part I (use typical best: lr=0.01, batch_size=64, hidden_sizes=[256,128])
BEST_HIDDEN = [256, 128]
BEST_LR = 0.01
BEST_BS = 64
EPOCHS_REG = 6

We define **train_model_reg**, which can train either **SimpleMLP** or **MLPWithBatchNorm** depending on `use_batchnorm`. It accepts **dropout_rate** and **weight_decay** (L2); the optimizer is Adam with that weight_decay. The training loop is the same as before: train one epoch (loss + accuracy), then evaluate on validation; at the end we compute **test accuracy**. We also set constants for Part II: **BEST_HIDDEN**, **BEST_LR**, **BEST_BS** (from Part I best config), and **EPOCHS_REG** (6 epochs for regularization experiments).

In [ ]:
# Create loaders once for Part II
train_loader_r = DataLoader(train_subset, batch_size=BEST_BS, shuffle=True, num_workers=0)
val_loader_r = DataLoader(val_subset, batch_size=BEST_BS, shuffle=False)
test_loader_r = DataLoader(test_dataset, batch_size=BEST_BS, shuffle=False)

# (a) Baseline (no regularization)
print("(a) Baseline – no regularization")
hist_baseline, test_baseline = train_model_reg(train_loader_r, val_loader_r, test_loader_r,
    BEST_LR, BEST_BS, BEST_HIDDEN, epochs=EPOCHS_REG, dropout_rate=0.0, weight_decay=0.0, use_batchnorm=False)

# (b) Dropout p=0.2 and p=0.5
print("(b) Dropout p=0.2")
hist_d02, test_d02 = train_model_reg(train_loader_r, val_loader_r, test_loader_r,
    BEST_LR, BEST_BS, BEST_HIDDEN, epochs=EPOCHS_REG, dropout_rate=0.2, weight_decay=0.0, use_batchnorm=False)
print("(b) Dropout p=0.5")
hist_d05, test_d05 = train_model_reg(train_loader_r, val_loader_r, test_loader_r,
    BEST_LR, BEST_BS, BEST_HIDDEN, epochs=EPOCHS_REG, dropout_rate=0.5, weight_decay=0.0, use_batchnorm=False)

# (c) L2 weight decay 1e-4 and 1e-3
print("(c) L2 weight_decay=1e-4")
hist_wd4, test_wd4 = train_model_reg(train_loader_r, val_loader_r, test_loader_r,
    BEST_LR, BEST_BS, BEST_HIDDEN, epochs=EPOCHS_REG, dropout_rate=0.0, weight_decay=1e-4, use_batchnorm=False)
print("(c) L2 weight_decay=1e-3")
hist_wd3, test_wd3 = train_model_reg(train_loader_r, val_loader_r, test_loader_r,
    BEST_LR, BEST_BS, BEST_HIDDEN, epochs=EPOCHS_REG, dropout_rate=0.0, weight_decay=1e-3, use_batchnorm=False)

# (d) BatchNorm + Dropout
print("(d) BatchNorm + Dropout")
hist_bn, test_bn = train_model_reg(train_loader_r, val_loader_r, test_loader_r,
    BEST_LR, BEST_BS, BEST_HIDDEN, epochs=EPOCHS_REG, dropout_rate=0.2, weight_decay=0.0, use_batchnorm=True)

We create data loaders for Part II, then train **six models**: (a) **Baseline**—no dropout, no weight decay; (b) **Dropout p=0.2** and **p=0.5**; (c) **L2 weight decay** 1e-4 and 1e-3; (d) **BatchNorm + Dropout** (using MLPWithBatchNorm with dropout 0.2). Each run returns history and test accuracy. All use the same best architecture and learning rate from Part I so we can fairly compare the effect of regularization.

### 2.3 Training Curves Side-by-Side & Overfitting Gap

- **Dropout**: Randomly zeros activations during training to prevent co-adaptation and overfitting.
- **L2 weight decay**: Penalizes large weights in the loss, encouraging smaller weights and smoother solutions.
- **BatchNorm**: Normalizes activations per batch; stabilizes training and has a mild regularizing effect.

In [ ]:
configs = [
    (hist_baseline, 'Baseline', test_baseline),
    (hist_d02, 'Dropout p=0.2', test_d02),
    (hist_d05, 'Dropout p=0.5', test_d05),
    (hist_wd4, 'L2 wd=1e-4', test_wd4),
    (hist_wd3, 'L2 wd=1e-3', test_wd3),
    (hist_bn, 'BatchNorm+Dropout', test_bn),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, EPOCHS_REG + 1)
for h, name, _ in configs:
    axes[0].plot(ep, h['train_loss'], label=name)
    axes[1].plot(ep, h['val_acc'], label=name)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss'); axes[0].set_title('Training Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy'); axes[1].set_title('Validation Accuracy')
axes[0].legend(); axes[1].legend()
axes[0].grid(True, alpha=0.3); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

 We collect all six configurations (baseline, two dropouts, two L2, BatchNorm+Dropout) and plot **training loss** (left) and **validation accuracy** (right) vs epoch. This side-by-side view shows how each regularizer affects **convergence** and **validation performance**; typically stronger regularization lowers train loss more slowly but can improve or stabilize val accuracy.

In [ ]:
# Overfitting gap: train_acc - val_acc (final epoch)
gap_data = []
for h, name, test_acc in configs:
    train_acc_f = h['train_acc'][-1]
    val_acc_f = h['val_acc'][-1]
    gap = train_acc_f - val_acc_f
    gap_data.append({'Config': name, 'Train Acc': train_acc_f, 'Val Acc': val_acc_f,
                     'Overfitting Gap': gap, 'Test Acc': test_acc})
df_gap = pd.DataFrame(gap_data)
print('Overfitting gap (Train Acc - Val Acc) and Test Accuracy:')
display(df_gap)

For each of the six configurations we compute the **overfitting gap** = (final train accuracy − final val accuracy). A large gap means the model fits the training set much better than the validation set (overfitting). We also record **test accuracy**. The table lets us compare which regularization reduces the gap and which achieves the best test performance.

---
## Data Augmentation Example

Before the exercises, we illustrate **data augmentation**: applying random transformations (rotation, flip, etc.) to training images so the model sees **varied versions** of each sample. This increases effective data diversity and helps **reduce overfitting** and improve generalization. Below we load one image and show how it looks under different augmentations.

In [ ]:
# Load one sample image (PIL) without normalization so we can apply augmentations and visualize
from PIL import Image
train_raw = datasets.MNIST(root='./data', train=True, download=True)  # no transform
img_pil, label = train_raw[0]  # PIL Image

# Define individual augmentations (we apply them one at a time for visualization)
aug_rotation = transforms.RandomRotation(10)
aug_flip = transforms.RandomHorizontalFlip(p=1.0)  # p=1 for demo: always flip
aug_affine = transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))  # slight translation

# Create augmented versions
img_rot = aug_rotation(img_pil)
img_flip = aug_flip(img_pil)
img_affine = aug_affine(img_pil)

# Optional: combined transform (as used in Exercise Part B)
transform_combined = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])
train_aug_demo = datasets.MNIST(root='./data', train=True, download=True, transform=transform_combined)
img_combined_1 = train_aug_demo[0][0]  # different each time due to randomness
img_combined_2 = train_aug_demo[0][0]  # fetch again to show another random view

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
axes[0, 0].imshow(img_pil, cmap='gray'); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
axes[0, 1].imshow(img_rot, cmap='gray'); axes[0, 1].set_title('RandomRotation(10°)'); axes[0, 1].axis('off')
axes[0, 2].imshow(img_flip, cmap='gray'); axes[0, 2].set_title('HorizontalFlip'); axes[0, 2].axis('off')
axes[1, 0].imshow(img_affine, cmap='gray'); axes[1, 0].set_title('Translate(0.1, 0.1)'); axes[1, 0].axis('off')
axes[1, 1].imshow(img_combined_1.squeeze(), cmap='gray'); axes[1, 1].set_title('Combined (view 1)'); axes[1, 1].axis('off')
axes[1, 2].imshow(img_combined_2.squeeze(), cmap='gray'); axes[1, 2].set_title('Combined (view 2)'); axes[1, 2].axis('off')
plt.suptitle('Data Augmentation Example (MNIST)', fontsize=12)
plt.tight_layout()
plt.show()

We load **one MNIST image** without normalization. We apply **RandomRotation(10)** (rotate by up to ±10°), **RandomHorizontalFlip** (flip with p=1 for demo), and **RandomAffine** with a small translation. We also show two samples from a dataset that uses the **combined transform** (RandomRotation + RandomHorizontalFlip + ToTensor + Normalize)—each time we access the same index we get a new random augmented view. The grid shows **original**, **rotated**, **flipped**, **translated**, and **two combined augmented views**. In training, such randomness is applied on-the-fly to each batch, so the model never sees the exact same augmented image twice.

---
## Exercise Section (Total 100 marks – 50 + 50)

### Exercise Part A – Hyperparameter Tuning (50 marks)

Use **Fashion-MNIST** instead of MNIST.

1. Load Fashion-MNIST the same way as MNIST (ToTensor + Normalize).
2. **Dataset Description & Visualization** (same style as Part I):
   - Print dataset size, number of classes (list the 10 clothing names), image shape.
   - Show 2×5 grid of sample images with class names.
   - Show one image per class in a grid.
   - Plot class distribution bar chart.
3. Tune: **learning rate**, **batch size**, **number of hidden layers** (2 vs 3), **hidden nodes per layer**.
4. Produce a results table and select the best model.
5. Save the best model as `best_fashion_model.pth`.

#### Part A – Load Fashion-MNIST and Dataset Description

In [ ]:
# Fashion-MNIST: same normalization as MNIST (same image stats)
transform_f = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])
train_f = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_f)
test_f = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_f)

# Validation split
n_train_f = len(train_f)
n_val_f = max(1, n_train_f // 10)
train_f_sub, val_f_sub = torch.utils.data.random_split(train_f, [n_train_f - n_val_f, n_val_f])

# Class names for Fashion-MNIST
FASHION_CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
num_classes_f = 10
sample_f, _ = train_f[0]
img_shape_f = sample_f.shape

print('=== Fashion-MNIST Dataset Description ===')
print(f'Training samples: {len(train_f)}, Validation: {len(val_f_sub)}, Test: {len(test_f)}')
print(f'Number of classes: {num_classes_f}')
print(f'Class names: {FASHION_CLASSES}')
print(f'Image shape (C, H, W): {img_shape_f}')
print(f'Flattened input size: {img_shape_f[0] * img_shape_f[1] * img_shape_f[2]}')

 We load **Fashion-MNIST** with the same transform as MNIST (ToTensor + Normalize). We split training into train/validation subsets. We define the **10 class names** (T-shirt/top, Trouser, etc.) and print dataset size, number of classes, class names, image shape, and flattened input size—mirroring the MNIST description in Part I.

In [ ]:
# 2×5 grid of sample images with class names
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.flatten()
for i in range(10):
    img, label = train_f[i]
    axes[i].imshow(img.squeeze(), cmap='gray')
    axes[i].set_title(FASHION_CLASSES[label], fontsize=9)
    axes[i].axis('off')
plt.suptitle('Fashion-MNIST – 10 Sample Images (2×5)', fontsize=12)
plt.tight_layout()
plt.show()

We display the **first 10 training images** of Fashion-MNIST in a 2×5 grid with **class names** (not numeric labels) as titles, in the same style as Part I.

In [ ]:
# One example per class
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.flatten()
for c in range(10):
    idx = next(i for i in range(len(train_f)) if train_f[i][1] == c)
    img, _ = train_f[idx]
    axes[c].imshow(img.squeeze(), cmap='gray')
    axes[c].set_title(FASHION_CLASSES[c], fontsize=9)
    axes[c].axis('off')
plt.suptitle('Fashion-MNIST – One Example per Class', fontsize=12)
plt.tight_layout()
plt.show()

For each of the 10 Fashion-MNIST classes we find one training example and show it in a 2×5 grid with the **class name** as title, giving one representative image per clothing type.

In [ ]:
# Class distribution bar chart
train_labels_f = [train_f[i][1] for i in range(len(train_f))]
counts_f = np.bincount(train_labels_f, minlength=10)
fig, ax = plt.subplots(figsize=(10, 4))
x_pos = np.arange(10)
bars = ax.bar(x_pos, counts_f, color=plt.cm.viridis(np.linspace(0.2, 0.8, 10)))
ax.set_xticks(x_pos)
ax.set_xticklabels(FASHION_CLASSES, rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Fashion-MNIST – Class Distribution (Training Set)')
for b, v in zip(bars, counts_f):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 50, str(v), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

We compute the **count per class** for Fashion-MNIST training set and plot a **bar chart** with class names on the x-axis (rotated for readability) and counts on the y-axis, with count annotations on top—same style as Part I.

#### Part A – Grid Search (Student implementation)

**Hints:**
- Define lists for **learning rate** (e.g. at least 2 values), **batch size** (e.g. 2–3 values), and **hidden layer configs**: at least two with **2 hidden layers** and at least one with **3 hidden layers** (e.g. `[128, 64]`, `[256, 128]`, `[256, 128, 64]`).
- Loop over all combinations. For each: create `DataLoader`s for `train_f_sub`, `val_f_sub`, and `test_f`; call **`train_model(...)`** (from Part I) with the current lr, batch_size, hidden_sizes, and a small number of epochs (e.g. 5–8).
- Append to a results list: lr, batch_size, hidden_sizes, final **validation accuracy** (`history['val_acc'][-1]`), and **test accuracy**.
- Build a **pandas DataFrame** from the results and display it. Select the **best** configuration (e.g. by maximum validation accuracy).

In [ ]:
# Hyperparameter search space
lr_list = [0.001, 0.01]
batch_size_list = [32, 64, 128]
hidden_sizes_list = [[128, 64], [256, 128], [256, 128, 64]]

epochs_f = 6
results_f = []
all_hist_f = {}

for lr in lr_list:
    for bs in batch_size_list:
        for hidden in hidden_sizes_list:

            key = (lr, bs, tuple(hidden))

            train_loader = DataLoader(train_f_sub, batch_size=bs, shuffle=True)
            val_loader = DataLoader(val_f_sub, batch_size=bs, shuffle=False)
            test_loader = DataLoader(test_f, batch_size=bs, shuffle=False)

            print(f"Running config: lr={lr}, batch_size={bs}, hidden={hidden}")

            history, test_acc, _ = train_model(
                train_loader,
                val_loader,
                test_loader,
                learning_rate=lr,
                batch_size=bs,
                hidden_sizes=hidden,
                epochs=epochs_f
            )

            all_hist_f[key] = history

            results_f.append({
                "lr": lr,
                "batch_size": bs,
                "hidden_sizes": str(hidden),
                "val_acc_final": history['val_acc'][-1],
                "test_acc": test_acc
            })

# Create dataframe
df_f = pd.DataFrame(results_f)

print("=== Grid Search Results ===")
display(df_f)

#### Part A – Save the best model (Student implementation)

**Hints:**
- From your results table, get the row with **maximum validation accuracy** (e.g. `df_f.loc[df_f['val_acc_final'].idxmax()]`).
- Extract **best_lr_f**, **best_bs_f**, and **best_hidden_f** from that row.

In [ ]:
# Find best configuration based on validation accuracy
best_row_f = df_f.loc[df_f['val_acc_final'].idxmax()]

best_lr_f = best_row_f['lr']
best_bs_f = int(best_row_f['batch_size'])
best_hidden_f = eval(best_row_f['hidden_sizes'])

print("Best Configuration:")
print("Learning Rate:", best_lr_f)
print("Batch Size:", best_bs_f)
print("Hidden Layers:", best_hidden_f)

### Exercise Part B – Regularization (50 marks)

Using the **best architecture from Part A**, train four versions:

1. **No regularization** (baseline)
2. **Dropout** (p=0.3)
3. **L2 weight decay**
4. **Dropout + BatchNorm + Data Augmentation** (RandomRotation 10°, RandomHorizontalFlip)

Compare **test accuracy** and **overfitting gap**. Write a short paragraph explaining which regularization worked best and why.

#### Part B – Data Augmentation and four regularization runs

**Hints (implement yourself):**

1. **Augmented dataset (for config 4):** Define a transform with **RandomRotation(10)**, **RandomHorizontalFlip()**, **ToTensor()**, **Normalize((MNIST_MEAN,), (MNIST_STD,))**. Create a Fashion-MNIST training dataset with this transform and split it into train/val (e.g. 90/10). Use this **only for training** in config 4; keep validation and test **without** augmentation for fair comparison.

2. **Four training runs:** Use the **best architecture from Part A** (best_lr_f, best_bs_f, best_hidden_f). Call **`train_model_reg`** (from Part II) for:
   - **(1) Baseline:** dropout_rate=0, weight_decay=0, use_batchnorm=False.
   - **(2) Dropout p=0.3:** dropout_rate=0.3, weight_decay=0, use_batchnorm=False.
   - **(3) L2 weight decay:** dropout_rate=0, weight_decay=1e-4 (or similar), use_batchnorm=False.
   - **(4) Dropout + BatchNorm + Augmentation:** use the **augmented** training DataLoader, dropout_rate=0.3, use_batchnorm=True.

3. **Comparison table:** Build a list of (history, config_name, test_acc) for each run. From each history compute final train acc, val acc, and **overfitting gap** = train_acc − val_acc. Put these in a pandas DataFrame and display it. Use this to write your short paragraph on which regularization worked best.

In [1]:
augment_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

# Augmented dataset
train_aug_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=augment_transform
)

# Split 90/10
n_train = len(train_aug_dataset)
n_val = n_train // 10
n_train_use = n_train - n_val

train_aug_sub, val_aug_sub = torch.utils.data.random_split(
    train_aug_dataset,
    [n_train_use, n_val]
)

train_aug_loader = DataLoader(train_aug_sub, batch_size=best_bs_f, shuffle=True)
val_aug_loader = DataLoader(val_aug_sub, batch_size=best_bs_f, shuffle

SyntaxError: incomplete input (4192734122.py, line 27)

#### Part B – Your Short Paragraph (Student Answer)

**Which regularization worked best and why?**

*[Your paragraph here. Consider: Did dropout reduce the overfitting gap? Did L2 improve test accuracy? Did Dropout+BatchNorm+Augmentation give the best generalization? Explain in 4–6 sentences.]*

---
## Key Takeaways

- **Hyperparameter tuning** (e.g. grid search over learning rate, batch size, and architecture) systematically improves model selection; validation metrics should guide the choice of the best configuration.
- **Regularization** techniques (Dropout, L2 weight decay, BatchNorm, data augmentation) reduce overfitting and often improve test accuracy; the overfitting gap (train − val accuracy) is a useful diagnostic.